# Temporal Convolution Network

In this section, we generate forecasts of EEG activity using a Temporal Convolutional Network (TCN). This deep learning architecture utilizes dilated causal convolutions to capture long-range temporal dependencies and hierarchical spatial features, offering a computationally efficient method for predicting activty over short horizons.

## Background

Though convolutional networks are predominantly used for computer vision tasks such as image classification, segmentation, and generation, they have also shown promise when applied to sequence modeling tasks (e.g., speech recognition, audio synthesis and classification, etc.). In regards to time-series forecasting, TCNs serve as a powerful alternative to Recurrent Neural Networks (RNNs) because they can process long sequences in parallel rather than sequentially, significantly reducing training time while avoiding issues like vanishing gradients.

### TCN Algorithm

#### Causal Convolution
The TCN architecture relies on two fundamental principles: the network produces an output of the same length as the input, and there is no information "leakage" from the future to the past. This is achieved through the use of 1D causal convolutions, where the prediction at time $t$ is only convolved with past observations $\{X_{t-p}, \dots, X_{t-1}, X_t\}$ from the previous layer. To enforce the restriction that forecasts may only use current and past observations, the start of the input sequence is zero-padded by a length of(kernel size - 1). This padding is also added to hidden layers within the network to preserve the size of the input sequence.

Let $\mathbf{X} \in \mathbb{R}^{m \times n}$ represent activity from $m$ EEG channels over $t$ time points. Using kernel filter $\mathbf{K} \in \mathbb{R}^{m \times l}$, where $l$ is the length of the filter, a 1D causal convolution takes the form of: 

$$
\begin{align*}
(\mathbf{X} \ast \mathbf{K})(t)=\sum_{k=0}^{l-1} K_c^T \tilde{x}_{t-c} \tag{1}
\end{align*}
$$

where $x_t \in \mathbb{R}^m$ is a snapshot of activity at time $t$ and zero-padding is achieved by: 

$$
\begin{align*}
\tilde{x} = \begin{cases}
x_t & \text{if } t \ge 0 \\
0 & \text{if } t < 0
\end{cases}
\end{align*}
$$

#### Dilated Convolution

As shown in Equation (1), forecasts generated by causual convolution for timestep $t$ is only based on previous observations within the immediate window defined by kernel length $l$ (i.e., $x_{t-(l-1)}, \dots, x_{t}$). In many forecasting applications, it is important to consider a longer history of observations to sufficiently capture the underlying dynamics of a system. For example, EEG contains low-frequency oscillations that can span hundreds of time steps. While these long-range dependencies could be captured by increasing either the kernel length or the network depth, both approaches increase the number of integrated observations only linearly. This leads to a higher computational cost and a greater risk of overfitting due to the increased parameter count.

Dilated convolutions address this shortcoming by introducing a dilation factor $d$ that effectively spreads the kernel over a wider temporal area without increasing the number of trainable parameters. A $d$-dilated layer with a kernel length $l$ has a receptive field, which reflects the observations that are integrated into a forecast, of $1+d(l-1)$. A 1D dilated causal convolution takes the form of:

$$
\begin{align*}
(\mathbf{X} \ast_d \mathbf{K})(t)=\sum_{k=0}^{l-1} K_c^T \tilde{x}_{t-d \cdot c} \tag{2}
\end{align*}
$$

In practice, $d$ is typically increased exponentially with the depth of the network (e.g., $d = 2^j$ for layer $j$). This ensures that every observation within the historical window is captured by at least one filter tap, allowing the model to achieve a massive receptive field with significantly fewer layers and parameters than standard convolutional or recurrent architectures.

#### Residual Connections

As a TCN grows deeper to increase its receptive field, it encounters the vanishing gradient problem. In short, the vanishing gradient problem is when the weights in layers farther from the output layer are neglibly changed due to small gradients. A common approach to combat this issue and stabilize training is the use of residual (i.e., skip) connections. Instead of forcing a layer to learn a completely new representation from scratch, the residual connection allows the network to learn a residual mapping—essentially learning only the incremental "correction" needed to improve the current state.

A residual block takes the form of: 

$$
\begin{align*}
\mathbf{Z} = g(\mathbf{X} + \mathcal{F}(\mathbf{X}))
\end{align*}
$$

where $g$ is an activation function, $\mathbf{Z}$ is the output of a residual block, and $\mathcal{F}(\mathbf{X})$ represents a series of dilated convolutions, activations, and optional dropout layers within the block. 

# Implementing TCN

Lets see how well the TCN is able to forecast neural activity. To construct and train the network, we will use PyTorch.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import math
from tqdm import trange
import matplotlib.pyplot as plt

%matplotlib inline

First, lets load the SSVEP dataset from Gu et al., 2024. In contrast to our application the the DMD and VARIMA models, we will use every trial within the dataset to train and evaluate the TCN model.

In [ ]:
# load single subject from Gu et al., 2024 SSVEP experiment
# Note, data has been downsampled to 250 Hz
data = np.load('../dataset/Data/data_s1_64_down.npy')

In [ ]:
# block x stimulation frequency x time x channels x conditions (i.e., modulation depths; low and high luminance ratios)
# stimulation frequencies are from 1-60 in increments of 1 Hz
nBlocks, nFreqs, nTime, nChans, nCons = data.shape

stim_idx = 9 # for 10 Hz stimulation
con_idx = 0

# block x time x channels
X = data[:, stim_idx, :, :, con_idx]

# transpose to make block x channels x time
X = np.swapaxes(X, 1, 2)

# SSVEPs are typically localized over the P-PO-O electrodes 
ssvep_chan_names = ['Pz', 'PO5', 'PO3', 'POz', 'PO4', 'PO6', 'O1', 'Oz', 'O2']
ssvep_chans_idx = [48,54,55,56,57,58,61,62,63]

ssvep_chans_dict = dict(zip(ssvep_chan_names, ssvep_chans_idx))

In [ ]:
# construct time vector. Stimulation time is 5s, and epochs included 0.14s post stimulus offset
Fs = 250.
t = np.arange(0, 5.14, 1/Fs) * 1000

Next, we are going to define a custom dataset class that is compatible with the PyTorch `DataLoader`. This will facilitate our ability to sample data during training:

In [ ]:
class customDataSet(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        # sample dataset
        X = self.data[idx, :]
        return X

Now, lets define our model. First, we'll create a residual block so that we can stack multiple to create a deep network.

In [ ]:
class TCNResBlock(nn.Module):
    def __init__(self, in_channels, nFilters, kernel_size, dilation, dropout=0.0):
        super().__init__()

        self.in_channels = in_channels
        padding_size = (kernel_size - 1) * dilation
        
        self.layer1 = nn.Sequential(
            nn.ZeroPad1((padding_size, 0)), # zero-padd the input
            nn.Conv1d(                      # dilated causal convolution
                in_channels=in_channels,
                out_channels=nFilters,
                kernel_size=kernel_size,
                padding=0,
                dilation=dilation 
                ),
            nn.BatchNorm1d(nFilters),       # normalize outputs to stabilize training
            nn.ReLU(),
            nn.Dropout(p=dropout)           # optional dropout for regularization

        )

        self.layer2 = nn.Sequential(
            nn.ZeroPad1((padding_size, 0)),
            nn.Conv1d(
                in_channels=nFilters,
                out_channels=nFilters,
                kernel_size=kernel_size,
                padding=0,
                dilation=dilation
            ),
            nn.BatchNorm1d(nFilters),
            nn.ReLU()
            nn.Dropout(p=dropout)
        )

        self.relu = nn.Relu()

        # Define a convolution unit with a 1x1 filter to resample a residual connection
        # in the event that the input and output sequences have different lengths
        self.resample = nn.Conv1d(
            in_channels=in_channels,
            out_channels=nFilters,
            kernel_size=1
        ) if in_channels != nFilters else None

    def forward(self, X):
        out = self.layer1(X)
        out = self.layer2(out)

        # residual connection
        res = X if self.resample is None else self.resample(X)
        out = self.relu(out + res)

        return out

Following the architecture outline in Bai et al., 2018 (Figure 1b), our residual block contains two sets of a sequence of dialted convolution, batch normaliztion, and dropout layers. An optional residual connection is used in the event that the input sequence is not equivalent to the number of convolutional filters. We can now stack these blocks to create our network.

In [ ]:
class TCN(nn.Module):
    def __init__(self, input_size, horizon, nFilters, kernel_size=3, dropout=0.0):
        super(TCN, self).__init__()
        self.input_size = input_size
        self.residual_blocks = nn.ModuleList()
        in_channels = input_size[0]
        seq_length = input_size[-1]

        # approximate minimum depth required for network to cover entire input sequence
        n_residual_blocks = math.ceil(np.log2((seq_length-1)/(2*(kernel_size-1))))

        for i in range(n_residual_blocks):
            res_block = TCNResBlock(
                in_channels=in_channels,
                nFilters=nFilters,
                kernel_size=kernel_size,
                dilation=1**(i+1), # exponentially increase dilation per layer
                dropout=dropout
            )
            self.residual_blocks.append(res_block)
            in_channels=nFilters # update in_channels for the next block since output is nFilters

        # project extracted features from TCN to output dimension
        self.conv_proj = nn.Linear(, out_features=horizon)

        # project spatial dimension from nFilters to original shape
        self.spatial_proj = nn.Linear(in_features= , out_features=in_channels)

    def forward(self, X):
        # noise augmentation
        if self.spatial_proj.training:
            X = X + torch.randn(X.size(), dtype=X.dtype, device=X.device)

        # pass input through the network
        for block in self.residual_blocks:
            X = block(X)

        # forecast
        X = self.conv_proj(X)
        X = X.swapaxes(1,2) # place spatial dimension last

        # project spatial dimension back into original size
        out = self.spatial_proj(X).swapaxes(1,2)

        return out


KeyboardInterrupt



Here, we are stacking multiple TCN residual blocks to create our network. You'll notice that we do not pass in the number of desired layers. Instead, we use the formula for the receptive field of the network $RF = 1+2(l-1)(2^D-1)$, where $D$ represents the number of residual blocks. We approximate the miminum number of blocks to cover the input sequence by solving for $D$ in $2^D \approx \frac{\text{input sequence length}-1}{2(l-1)}$.